# Basic operations

## 1. Создание датафрейма views

In [39]:
import pandas as pd

input_file = "../data/feed-views.log"

views = pd.read_csv(
    input_file, 
    sep='\t',
    header=None, 
    names=["datetime", "user"]
    )

views

,datetime,user
0,2020-04-17 12:01:08.463179,artem
1,2020-04-17 12:01:23.743946,artem
2,2020-04-17 12:27:30.646665,artem
3,2020-04-17 12:35:44.884757,artem
4,2020-04-17 12:35:52.735016,artem
...,...,...
1071,2020-05-21 18:45:20.441142,valentina
1072,2020-05-21 23:03:06.457819,maxim
1073,2020-05-21 23:23:49.995349,pavel
1074,2020-05-21 23:49:22.386789,artem


- Конвертирование datetime в datetime64[ns]

In [40]:
views["datetime"] = views["datetime"].astype("datetime64[ns]")

views.dtypes

datetime    datetime64[ns]
user                object
dtype: object

- Создание новых столбцов year, month, day, hour, minute, second

In [41]:
views["year"] = views["datetime"].dt.year
views["month"] = views["datetime"].dt.month
views["day"] = views["datetime"].dt.day
views["hour"] = views["datetime"].dt.hour
views["minute"] = views["datetime"].dt.minute
views["second"] = views["datetime"].dt.second

views.head()

,datetime,user,year,month,day,hour,minute,second
0,2020-04-17 12:01:08.463179,artem,2020,4,17,12,1,8
1,2020-04-17 12:01:23.743946,artem,2020,4,17,12,1,23
2,2020-04-17 12:27:30.646665,artem,2020,4,17,12,27,30
3,2020-04-17 12:35:44.884757,artem,2020,4,17,12,35,44
4,2020-04-17 12:35:52.735016,artem,2020,4,17,12,35,52


## 2. Создание новых столбцов daytime

- Присваивание значения времени суток к интервалам

In [42]:
time_bins = [0, 4, 7, 11, 17, 20, 24]
time_labels = ["night", "early morning", "morning", "afternoon", "early evening", "evening"]

views["daytime"] = pd.cut(views["hour"], bins=time_bins, labels=time_labels, right=False)

- Установка user в качестве индексного столбца

In [43]:
views.set_index("user", inplace=True)

views

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
artem,2020-04-17 12:01:08.463179,2020,4,17,12,1,8,afternoon
artem,2020-04-17 12:01:23.743946,2020,4,17,12,1,23,afternoon
artem,2020-04-17 12:27:30.646665,2020,4,17,12,27,30,afternoon
artem,2020-04-17 12:35:44.884757,2020,4,17,12,35,44,afternoon
artem,2020-04-17 12:35:52.735016,2020,4,17,12,35,52,afternoon
...,...,...,...,...,...,...,...,...
valentina,2020-05-21 18:45:20.441142,2020,5,21,18,45,20,early evening
maxim,2020-05-21 23:03:06.457819,2020,5,21,23,3,6,evening
pavel,2020-05-21 23:23:49.995349,2020,5,21,23,23,49,evening


## 3.Расчет кол-ва элементов во всем датаврейме

In [44]:
views.count()


datetime    1076
year        1076
month       1076
day         1076
hour        1076
minute      1076
second      1076
daytime     1076
dtype: int64

- Раcчет кол-ва элементов в каждой категории времени суток

In [45]:
views["daytime"].value_counts()

daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64

## 4.Сортировка по часам, минутам и секундам одновременно

In [46]:
views.sort_values(by=["hour", "minute", "second"])

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
valentina,2020-05-15 00:00:13.222265,2020,5,15,0,0,13,night
valentina,2020-05-15 00:01:05.153738,2020,5,15,0,1,5,night
pavel,2020-05-12 00:01:27.764025,2020,5,12,0,1,27,night
pavel,2020-05-12 00:01:38.444917,2020,5,12,0,1,38,night
pavel,2020-05-12 00:01:55.395042,2020,5,12,0,1,55,night
...,...,...,...,...,...,...,...,...
artem,2020-05-21 23:49:22.386789,2020,5,21,23,49,22,evening
anatoliy,2020-05-09 23:53:55.599821,2020,5,9,23,53,55,evening
pavel,2020-05-09 23:54:54.260791,2020,5,9,23,54,54,evening


## 5. Расчет минимума и максимума для часов, а также режим для дневных категорий

In [47]:
min_hour = views["hour"].min()
max_hour = views["hour"].max()
min_mode = views["daytime"].min()
max_mode = views["daytime"].max()

print(f"min_hour: {min_hour}\nmax_hour: {max_hour}\nmin_mode: {min_mode}\nmax_mode: {max_mode}")

min_hour: 0
max_hour: 23
min_mode: night
max_mode: evening


- Расчет максимального часа для строк, где время суток — ночь

In [48]:
max_hour_night = views[views["daytime"] == "night"]["hour"].max()

print(max_hour_night)

3


- Расчет минимального часа для строк, где время суток — утро

In [49]:
min_hour_morning = views[views["daytime"] == "morning"]["hour"].min()

print(min_hour_morning)

8


- Вывод, кто посещал страницу в ночью

In [50]:
user_max_hour_night = views[views["daytime"] == "night"]["hour"].idxmax()


print(user_max_hour_night)


konstantin


- Вывод, кто посещал страницу в утром

In [51]:
user_min_hour_morning = views[views["daytime"] == "morning"]["hour"].idxmin()

print(user_min_hour_morning)

alexander


- Расчет моды для часа

In [52]:
mode_hour = views["hour"].mode()
mode_hour[0]

np.int32(22)

 - Расчет модs для времени суток

In [53]:
mode_daytime = views["daytime"].mode()
mode_daytime[0]

'evening'

## 6. Поиск трех самых ранних и трех самых поздних часа дня и соответствующие им имена пользователей с помощью nsmallest() и nlargest()

- 3 самых ранних часа

In [54]:
views.nsmallest(3, "hour")[["hour"]]

,hour
user,
artem,0
konstantin,0
konstantin,0


- 3 самых поздних часа

In [55]:
views.nlargest(3, "hour")[["hour"]] 

,hour
user,
konstantin,23
artem,23
artem,23


## 7. Вычисление межквартильного размаха (IQR)

- Общая статистика по датафрейму

In [56]:
stats = views.describe()
stats

,datetime,year,month,day,hour,minute,second
count,1076,1076.0,1076.000000,1076.000000,1076.000000,1076.000000,1076.000000
mean,2020-05-10 09:00:41.211420672,2020.0,4.870818,13.552974,16.249071,29.629182,29.500929
min,2020-04-17 12:01:08.463179,2020.0,4.000000,1.000000,0.000000,0.000000,0.000000
25%,2020-05-10 01:13:49.857472,2020.0,5.000000,11.000000,13.000000,14.000000,14.000000
50%,2020-05-11 22:48:35.302552832,2020.0,5.000000,13.000000,19.000000,29.000000,30.000000
75%,2020-05-14 14:44:34.749530624,2020.0,5.000000,15.000000,22.000000,46.000000,45.000000
max,2020-05-22 10:36:14.662600,2020.0,5.000000,30.000000,23.000000,59.000000,59.000000
std,NaN,0.0,0.335557,4.906567,6.955490,17.689388,17.405506


- Межквартильный размах (IQR)

In [57]:
iqr = stats.loc['75%'][["hour"]] - stats.loc['25%'][["hour"]]

iqr["hour"]

np.float64(9.0)